# DV-E23: High-Engagement Video Filtering

| What to expect | Value |
| --- | --- |
| Scenario | Content platform |
| Difficulty | Easy |
| Delivery scope | Technique drill |
| Expected effort | 30-45 minutes |
| Deliverable | One pre-created Python transformation file |
| Primary interface | PySpark DataFrame API |
| Prerequisites | DataFrame basics and Python functions |
| Tooling | `python`, `pyspark`, `spark` |
| Techniques | `explicit-schema`, `filter`, `projection`, `deterministic-order`, `dataframe-test` |

## Start here

1. Read the problem and schema below.
2. [Complete `build_high_engagement_videos` in the pre-created learner file](../../../src/big_data_example/labs/content_platform/dv_e23_high_engagement_video_filtering.py).
3. Return here and run Checkpoints 2 and 3 to reload and check your saved work.
4. After your attempt, open the [solution and explanation](../../../docs/interactive-data-engineering-labs/solutions/content-platform/dv-e23-high-engagement-video-filtering.md) (**spoiler**).

See the [lab README](../README.md) for environment setup, terminal test commands, and ways to restart or reset your work.

## 1. Problem

The content merchandising team needs a small candidate dataset of recent, highly viewed videos. Return metadata records with strictly more than 1,000,000 views and a release year of 2019 or later.

Output columns, in order: `duration`, `genre`, `release_year`, `title`, `video_id`, `view_count`. Order by `duration` ascending and then `video_id` ascending so equal-duration rows are deterministic.

The input is synthetic structured metadata. No video or other media file is required.

## 2. Schema and contract

Input grain: one metadata record per `video_id` in this delivery. Null `view_count` or `release_year` values do not qualify.

| Input column | Spark type | Meaning |
| --- | --- | --- |
| `video_id` | `IntegerType` | Stable video identifier |
| `title` | `StringType` | Display title |
| `genre` | `StringType` | Content category |
| `release_year` | `IntegerType` | Calendar release year |
| `duration` | `IntegerType` | Duration in minutes |
| `view_count` | `LongType` | Accumulated view count |

Output schema, in contractual order: `duration`, `genre`, `release_year`, `title`, `video_id`, `view_count`.

<details>
<summary><strong>How the files fit together</strong></summary>

The notebook is the guide and runner. Your work belongs in the [pre-created DV-E23 learner file](../../../src/big_data_example/labs/content_platform/dv_e23_high_engagement_video_filtering.py), which currently contains a starter implementation.

Every lab will link its pre-created work files. A short drill may use one Python or SQL file; a pipeline lab may link several Python, SQL, shell, configuration, or infrastructure files. The notebook coordinates those artifacts and provides feedback.

Other artifacts:

- [Synthetic source data](../../../data/samples/interactive-data-engineering-labs/content-platform/source/batch-001/videos.csv)
- [Shared notebook setup](../../../src/big_data_example/notebook_support.py)
- [Solution and engineering explanation](../../../docs/interactive-data-engineering-labs/solutions/content-platform/dv-e23-high-engagement-video-filtering.md) (**spoiler**)

The source and expected fixtures are immutable exercise inputs.

</details>

<details>
<summary><strong>Run, restart, clear, or reset the lab</strong></summary>

- Run from a clean state: in VS Code choose **Restart** for the notebook kernel, then **Run All**.
- Clear displayed results: use **Clear All Outputs** from the notebook toolbar or Command Palette. This does not change Python files.
- Stop Spark without restarting Python: run the final cleanup cell.
- Reset this exercise after the lab baseline is committed: close the learner file, review your changes with `git diff -- src/big_data_example/labs/content_platform/dv_e23_high_engagement_video_filtering.py`, then restore that one file as shown in Section 8.

This technique drill does not write generated datasets, so there is no output directory to clean. Later stateful pipeline labs will provide a lab-specific reset command.

</details>

In [ ]:
import importlib
import sys

from big_data_example.labs.content_platform.dv_e23_check import (
    check_high_engagement_videos,
)
from big_data_example.labs.content_platform.dv_e23_contract import (
    VIDEO_INPUT_SCHEMA,
    read_video_metadata,
)
from big_data_example.notebook_support import local_spark, project_path

spark = local_spark("dv-e23-high-engagement-video-filtering", threads=2)
spark.sparkContext.setLogLevel("ERROR")
print(f"Environment ready: Python {sys.version.split()[0]}, Spark {spark.version}")

## 3. Checkpoint 1: Inspect the source delivery and schema

<details>
<summary>Open checkpoint instructions</summary>

Confirm that the declared schema represents identifiers, years, durations, and counts with appropriate Spark types. Run the next cell; acceptance evidence is a declared schema and ten input rows.

CSV is an untyped transport. The question-specific reader applies an explicit schema so the result does not depend on type inference from whichever sample values happen to be present. The fixture also contains threshold, null, old-content, and equal-duration cases that can expose plausible mistakes.

**Job connection:** Production pipelines receive data through contracts, not trustworthy notebook variables. Inspecting grain, types, nullability, and boundary records before transforming data is how engineers avoid silently publishing plausible but incorrect tables.

</details>

In [ ]:
input_path = project_path(
    "data",
    "samples",
    "interactive-data-engineering-labs",
    "content-platform",
    "source",
    "batch-001",
    "videos.csv",
)
video_stream_df = read_video_metadata(spark, input_path)

assert video_stream_df.schema == VIDEO_INPUT_SCHEMA
assert video_stream_df.count() == 10
video_stream_df.printSchema()
video_stream_df.orderBy("video_id").show(truncate=False)

## 4. Checkpoint 2: Complete the transformation

<details>
<summary>Open checkpoint instructions and Easy hint</summary>

[Complete `build_high_engagement_videos(video_stream_df)` in the learner file](../../../src/big_data_example/labs/content_platform/dv_e23_high_engagement_video_filtering.py). Use built-in Spark column expressions, preserve the exact output projection, and make ordering deterministic. Save that file, then run the next code cell.

Combine the strict view predicate and inclusive release-year predicate with parenthesized `Column` expressions and `&`. Apply `select` and `orderBy` after the filter.

**Job connection:** Production transformation logic belongs in versioned, testable Python or SQL rather than only in an exploratory notebook. This exercise keeps data inspection and feedback interactive while placing the deliverable in a normal Python module.

</details>

In [ ]:
from big_data_example.labs.content_platform import (
    dv_e23_high_engagement_video_filtering as exercise,
)

# Reload the learner module so this cell uses your latest saved changes.
exercise = importlib.reload(exercise)
actual_df = exercise.build_high_engagement_videos(video_stream_df)
actual_df.show(truncate=False)

## 5. Checkpoint 3: Validate the contract

<details>
<summary>Open checkpoint instructions</summary>

Run the next **Check my work** cell. It validates the DataFrame created by your notebook function. It is not the environment check and does not require you to open or edit a test file.

The checker reports whether the result satisfies the problem contract without displaying its implementation. A visually plausible `show()` result is useful for debugging but is not correctness proof.

**Job connection:** Data engineers validate schemas, boundaries, counts, and representative records because a job completing successfully only proves that it ran; it does not prove that it published correct data.

</details>

In [ ]:
check_high_engagement_videos(spark, actual_df)

## 6. Inspect the plan

<details>
<summary>Open plan instructions and question</summary>

Run the next cell and locate the filter, projection, and ordering operations.

Which operation requires data movement across partitions, and would you retain that operation when writing a large reusable dataset?

**Job connection:** Reviewing a Spark plan connects correct code to runtime behavior. A transformation that is cheap for ten rows can create a cluster-wide shuffle, extra latency, and substantial cost at production scale.

</details>

In [ ]:
actual_df.explain("formatted")

## 7. Reflection and solution

<details>
<summary>Reflection questions</summary>

1. Why is exactly 1,000,000 excluded while release year 2019 is included?
2. What happens to rows whose view count or release year is null?
3. Why is `view_count` represented as a long rather than a 32-bit integer?
4. Why can global ordering be expensive in a distributed job?

</details>

After your attempt, review the [solution and explanation](../../../docs/interactive-data-engineering-labs/solutions/content-platform/dv-e23-high-engagement-video-filtering.md) (**spoiler**).

## 8. Finish or reset the lab

<details>
<summary>Open cleanup and reset instructions</summary>

Run the next cell when finished to stop this notebook's Spark session. To clear runtime state without erasing saved work, use **Restart** and rerun the cells you need.

To discard your implementation and restore the committed starter, close the learner Python file and run this from a VS Code terminal opened at the repository root:

```powershell
git restore -- src/big_data_example/labs/content_platform/dv_e23_high_engagement_video_filtering.py
```

</details>

In [ ]:
spark.stop()

## 9. Definition of done

- The notebook runs from a restarted kernel using the project `.venv`.
- The linked learner implementation contains no Python UDF or schema inference.
- The notebook's **Check my work** cell passes.
- You can explain the strict/inclusive boundaries, null behavior, tie-break, and global-sort cost.